# OmniCare Clinical Copilot - Notebook 2: Admission & Vitals

**Pipeline:** Synthea FHIR Data → HAPI FHIR Server → Vitals Extraction → MedGemma → Admission Note

**Prerequisites:**
- Run `00_setup_and_models.ipynb` (models loaded)
- Run `01_consultation_audio_soap.ipynb` (consultation SOAP note created)
- HAPI FHIR server deployed on GCP Cloud Run (or use local parsing mode)

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from utils.encounter_state import load_encounter, update_stage, list_encounters
from utils.fhir_helpers import (
    parse_fhir_bundle, extract_vitals, extract_conditions,
    extract_medications, extract_patient_demographics,
    format_vitals_summary, upload_bundle_to_fhir, query_fhir_vitals
)
from utils.mcp_tools import generate_admission_note
from utils.prompts import ADMISSION_SYSTEM_PROMPT, ADMISSION_USER_TEMPLATE

# Configuration
USE_FHIR_SERVER = True  # Set to False for local-only parsing (no HAPI FHIR needed)
FHIR_SERVER_URL = ""  # Fill in after deploying HAPI FHIR on Cloud Run

# Encounter from previous notebook
available = list_encounters()
print(f"Available encounters: {available}")

# Use the most recent encounter, or set manually
encounter_id = available[-1] if available else None
print(f"Using encounter: {encounter_id}")

## 2. Deploy HAPI FHIR Server (GCP Cloud Run)

Run this once to deploy the FHIR server. After deployment, copy the URL into `FHIR_SERVER_URL` above.

In [ ]:
# Uncomment and run to deploy HAPI FHIR on GCP Cloud Run
# Requires: gcloud CLI authenticated

# !gcloud auth login
# !gcloud config set project YOUR_PROJECT_ID
#
# !gcloud run deploy hapi-fhir \
#   --image hapiproject/hapi:latest \
#   --platform managed \
#   --region us-central1 \
#   --allow-unauthenticated \
#   --port 8080 \
#   --memory 1Gi \
#   --cpu 1 \
#   --min-instances 0 \
#   --max-instances 1
#
# After deployment, get the URL:
# !gcloud run services describe hapi-fhir --region us-central1 --format 'value(status.url)'

print("HAPI FHIR deployment instructions ready.")
print("After deploying, set FHIR_SERVER_URL in cell 1.")

## 3. Generate or Load Synthea FHIR Data

In [ ]:
# Option A: Generate fresh Synthea data (requires Java)
# !apt-get install -y default-jre -qq
# !wget -q https://github.com/synthetichealth/synthea/releases/download/v3.3.0/synthea-with-dependencies.jar
# !java -jar synthea-with-dependencies.jar -p 1 Massachusetts --exporter.fhir.export=true

# Option B: Use a sample FHIR bundle (embedded for demo)
SAMPLE_BUNDLE = {
    "resourceType": "Bundle",
    "type": "collection",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "patient-001",
                "name": [{"given": ["John"], "family": "Smith"}],
                "birthDate": "1965-03-15",
                "gender": "male",
                "address": [{"line": ["123 Main St"], "city": "Boston", "state": "MA", "postalCode": "02101"}]
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "status": "final",
                "category": [{"coding": [{"code": "vital-signs"}]}],
                "code": {"coding": [{"code": "8310-5", "display": "Body temperature"}]},
                "valueQuantity": {"value": 99.8, "unit": "degF"},
                "effectiveDateTime": "2026-03-25T08:00:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "status": "final",
                "category": [{"coding": [{"code": "vital-signs"}]}],
                "code": {"coding": [{"code": "85354-9", "display": "Blood pressure panel"}]},
                "component": [
                    {
                        "code": {"coding": [{"display": "Systolic"}]},
                        "valueQuantity": {"value": 138, "unit": "mmHg"}
                    },
                    {
                        "code": {"coding": [{"display": "Diastolic"}]},
                        "valueQuantity": {"value": 82, "unit": "mmHg"}
                    }
                ],
                "effectiveDateTime": "2026-03-25T08:00:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "status": "final",
                "category": [{"coding": [{"code": "vital-signs"}]}],
                "code": {"coding": [{"code": "8867-4", "display": "Heart rate"}]},
                "valueQuantity": {"value": 88, "unit": "bpm"},
                "effectiveDateTime": "2026-03-25T08:00:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "status": "final",
                "category": [{"coding": [{"code": "vital-signs"}]}],
                "code": {"coding": [{"code": "9279-1", "display": "Respiratory rate"}]},
                "valueQuantity": {"value": 18, "unit": "breaths/min"},
                "effectiveDateTime": "2026-03-25T08:00:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "status": "final",
                "category": [{"coding": [{"code": "vital-signs"}]}],
                "code": {"coding": [{"code": "2708-6", "display": "Oxygen saturation (SpO2)"}]},
                "valueQuantity": {"value": 96, "unit": "%"},
                "effectiveDateTime": "2026-03-25T08:00:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Condition",
                "clinicalStatus": {"coding": [{"code": "active"}]},
                "code": {"coding": [{"code": "44054006", "display": "Type 2 diabetes mellitus"}]},
                "onsetDateTime": "2018-06-15"
            }
        },
        {
            "resource": {
                "resourceType": "Condition",
                "clinicalStatus": {"coding": [{"code": "active"}]},
                "code": {"coding": [{"code": "38341003", "display": "Essential hypertension"}]},
                "onsetDateTime": "2015-01-20"
            }
        },
        {
            "resource": {
                "resourceType": "MedicationRequest",
                "status": "active",
                "medicationCodeableConcept": {"coding": [{"display": "Metformin 500mg"}]}
            }
        },
        {
            "resource": {
                "resourceType": "MedicationRequest",
                "status": "active",
                "medicationCodeableConcept": {"coding": [{"display": "Lisinopril 10mg"}]}
            }
        }
    ]
}

# Save sample bundle for processing
bundle_path = "/content/sample_data/sample_patient_bundle.json"
os.makedirs("/content/sample_data", exist_ok=True)
with open(bundle_path, "w") as f:
    json.dump(SAMPLE_BUNDLE, f, indent=2)

print(f"FHIR bundle saved to: {bundle_path}")

## 4. Parse FHIR Bundle & Extract Clinical Data

In [ ]:
# Parse the FHIR bundle
resources = parse_fhir_bundle(bundle_path)

print("FHIR Resources extracted:")
for rtype, rlist in resources.items():
    if rlist:
        print(f"  {rtype}: {len(rlist)}")

# Extract structured data
demographics = extract_patient_demographics(resources["patients"][0]) if resources["patients"] else {}
vitals = extract_vitals(resources["observations"])
conditions = extract_conditions(resources["conditions"])
medications = extract_medications(resources["medications"])

print(f"\nPatient: {demographics.get('name', 'Unknown')}")
print(f"DOB: {demographics.get('dob', 'Unknown')}")
print(f"Vitals: {len(vitals)} readings")
print(f"Conditions: {len(conditions)}")
print(f"Medications: {len(medications)}")

## 5. Upload to HAPI FHIR Server (Optional)

In [ ]:
if USE_FHIR_SERVER and FHIR_SERVER_URL:
    print(f"Uploading bundle to HAPI FHIR: {FHIR_SERVER_URL}")
    try:
        result = upload_bundle_to_fhir(bundle_path, FHIR_SERVER_URL)
        print(f"Upload successful: {result.get('resourceType', 'OK')}")

        # Query vitals back from the server to verify
        patient_id = demographics.get("mrn", "patient-001")
        server_vitals = query_fhir_vitals(patient_id, FHIR_SERVER_URL)
        print(f"Verified: {len(server_vitals)} vitals retrieved from FHIR server")
    except Exception as e:
        print(f"FHIR server error: {e}")
        print("Continuing with local parsing...")
else:
    print("Skipping FHIR server upload (USE_FHIR_SERVER=False or no URL).")
    print("Using locally parsed FHIR data.")

## 6. Display Vitals Summary

In [ ]:
vitals_summary = format_vitals_summary(vitals)
print("Current Vital Signs:")
print(vitals_summary)

print("\nActive Conditions:")
for c in conditions:
    print(f"  - {c['display']} (onset: {c['onset']}, status: {c['status']})")

print("\nMedications:")
for m in medications:
    print(f"  - {m['display']} ({m['status']})")

## 7. Generate Admission Note (MedGemma)

In [ ]:
# Load consultation SOAP note from previous stage
encounter = load_encounter(encounter_id)
soap_note = encounter["stages"]["consultation"].get("soap_note", {})
soap_text = "\n".join(f"{k.upper()}: {v}" for k, v in soap_note.items() if v)

if not soap_text:
    soap_text = "No consultation SOAP note available."

# Format inputs for MedGemma
demographics_str = f"Name: {demographics.get('name', 'N/A')}, DOB: {demographics.get('dob', 'N/A')}, Gender: {demographics.get('gender', 'N/A')}"
conditions_str = "\n".join(f"- {c['display']}" for c in conditions) or "None documented"
medications_str = "\n".join(f"- {m['display']}" for m in medications) or "None documented"

print("Generating admission note with MedGemma...")
admission_note = generate_admission_note(
    demographics=demographics_str,
    soap_note=soap_text,
    vitals=vitals_summary,
    conditions=conditions_str,
    medications=medications_str,
    allergies="No known drug allergies (NKDA)",
    model=medgemma_model,
    processor=medgemma_processor,
    max_new_tokens=1024
)

print("\n" + "="*60)
print("ADMISSION NOTE")
print("="*60)
print(admission_note)

## 8. Save to Encounter State

In [ ]:
encounter = update_stage(encounter_id, "admission", {
    "fhir_patient_id": demographics.get("mrn", "patient-001"),
    "vitals_history": vitals,
    "conditions": conditions,
    "medications": medications,
    "admission_note": admission_note
})

print(f"Encounter {encounter_id} updated with admission data.")
print(f"\nAdmission stage complete!")
print(f"Next: Run 03_radiology_dicom_imaging.ipynb with encounter_id = '{encounter_id}'")